In [ ]:
"""
Minimal Object Detection Evaluation Metric for Kaggle - mAP@[.50:.95]

This is the minimal version containing only the score() function required for Kaggle.
Uses the official pycocotools library for COCO-style mAP calculation.
"""

import pandas as pd
import numpy as np
import io
import sys
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


class ParticipantVisibleError(Exception):
    """Custom exception for errors that should be visible to competition participants."""
    pass


def score(
    solution: pd.DataFrame,
    submission: pd.DataFrame,
    row_id_column_name: str
) -> float:
    """
    Calculate mAP@[.50:.95] score for object detection predictions using pycocotools.
    
    This is the main scoring function for Kaggle competitions. It uses the official
    COCO evaluation toolkit (pycocotools) to calculate mAP, averaging across IoU 
    thresholds from 0.50 to 0.95 in steps of 0.05.
    
    Parameters
    ----------
    solution : pd.DataFrame
        Ground truth annotations with columns:
        - row_id_column_name (e.g., 'annotation_id'): Unique row identifier (required by Kaggle)
        - image_id: Unique identifier for each image
        - category_id: Object category/class identifier
        - bbox_x, bbox_y, bbox_width, bbox_height: Bounding box coordinates in COCO format
    submission : pd.DataFrame
        Predicted annotations with columns:
        - row_id_column_name (e.g., 'annotation_id'): Unique row identifier (required by Kaggle)
        - image_id: Unique identifier for each image
        - category_id: Object category/class identifier
        - bbox_x, bbox_y, bbox_width, bbox_height: Bounding box coordinates in COCO format
        - score: Confidence score for the prediction (0-1)
    row_id_column_name : str
        Name of the column containing unique row identifiers (typically 'annotation_id').
        This is required by Kaggle's competition framework for both solution and submission.
    
    Returns
    -------
    float
        Mean Average Precision averaged over IoU thresholds [0.50:0.95:0.05]
        This is the official COCO mAP metric.
    
    Raises
    ------
    ParticipantVisibleError
        If required columns are missing from solution or submission files
    
    Notes
    -----
    - Uses official pycocotools COCOeval implementation
    - Evaluates at 10 IoU thresholds: 0.50, 0.55, 0.60, ..., 0.90, 0.95
    - Final score is the mean of all per-threshold mAP values
    - Bounding boxes should be in format [x, y, width, height]
    
    Examples
    --------
    >>> solution = pd.DataFrame({
    ...     'image_id': [1, 1, 2],
    ...     'category_id': [1, 2, 1],
    ...     'bbox_x': [10, 50, 20],
    ...     'bbox_y': [10, 50, 20],
    ...     'bbox_width': [30, 40, 25],
    ...     'bbox_height': [30, 40, 25]
    ... })
    >>> submission = pd.DataFrame({
    ...     'image_id': [1, 1, 2],
    ...     'category_id': [1, 2, 1],
    ...     'bbox_x': [12, 52, 22],
    ...     'bbox_y': [12, 52, 22],
    ...     'bbox_width': [28, 38, 23],
    ...     'bbox_height': [28, 38, 23],
    ...     'score': [0.9, 0.85, 0.95]
    ... })
    >>> mAP = score(solution, submission, 'id')
    
    References
    ----------
    .. [1] Lin, T. Y., Maire, M., Belongie, S., Hays, J., Perona, P., Ramanan, D., ...
           "Microsoft COCO: Common Objects in Context."
           European Conference on Computer Vision, 2014.
    .. [2] https://github.com/cocodataset/cocoapi
    """
    gt_df = solution.copy()
    pred_df = submission.copy()
    
    # Check required columns in solution
    required_solution_cols = ['image_id', 'category_id', 'bbox_x', 'bbox_y', 'bbox_width', 'bbox_height']
    for col in required_solution_cols:
        if col not in gt_df.columns:
            raise ParticipantVisibleError(f'Solution missing column: {col}')
    
    # Check required columns in submission
    required_submission_cols = ['image_id', 'category_id', 'bbox_x', 'bbox_y', 'bbox_width', 'bbox_height', 'score']
    for col in required_submission_cols:
        if col not in pred_df.columns:
            raise ParticipantVisibleError(f'Submission missing column: {col}')
    
    # Validate data types and values in submission
    numeric_cols = ['bbox_x', 'bbox_y', 'bbox_width', 'bbox_height', 'score']
    for col in numeric_cols:
        # Check for non-numeric values
        if not pd.api.types.is_numeric_dtype(pred_df[col]):
            raise ParticipantVisibleError(
                f'Column "{col}" must be numeric, got dtype: {pred_df[col].dtype}'
            )
        # Check for NaN values
        if pred_df[col].isna().any():
            nan_count = pred_df[col].isna().sum()
            raise ParticipantVisibleError(
                f'Column "{col}" contains {nan_count} NaN/null values. All values must be valid numbers.'
            )
        # Check for infinity values
        if np.isinf(pred_df[col]).any():
            inf_count = np.isinf(pred_df[col]).sum()
            raise ParticipantVisibleError(
                f'Column "{col}" contains {inf_count} infinity values. All values must be finite.'
            )
    
    # Validate score range [0, 1]
    if (pred_df['score'] < 0).any() or (pred_df['score'] > 1).any():
        invalid_scores = pred_df[(pred_df['score'] < 0) | (pred_df['score'] > 1)]['score']
        raise ParticipantVisibleError(
            f'Column "score" must be in range [0, 1]. Found {len(invalid_scores)} invalid values. '
            f'Min: {invalid_scores.min():.4f}, Max: {invalid_scores.max():.4f}'
        )
    
    # Validate positive bbox dimensions
    if (pred_df['bbox_width'] <= 0).any():
        invalid_count = (pred_df['bbox_width'] <= 0).sum()
        raise ParticipantVisibleError(
            f'Column "bbox_width" must be positive. Found {invalid_count} values <= 0.'
        )
    if (pred_df['bbox_height'] <= 0).any():
        invalid_count = (pred_df['bbox_height'] <= 0).sum()
        raise ParticipantVisibleError(
            f'Column "bbox_height" must be positive. Found {invalid_count} values <= 0.'
        )
    
    # Filter submission to only include images that exist in ground truth
    # This handles the case where the solution is a subset (e.g., Public/Private split)
    gt_image_ids = set(gt_df['image_id'].unique())
    pred_df = pred_df[pred_df['image_id'].isin(gt_image_ids)].copy()
    
    # If no predictions remain after filtering, return 0
    if len(pred_df) == 0:
        return 0.0
    
    # Validate category IDs - submission must only use categories that exist in ground truth
    # (after filtering to relevant images)
    valid_category_ids = set(gt_df['category_id'].unique())
    submission_category_ids = set(pred_df['category_id'].unique())
    invalid_category_ids = submission_category_ids - valid_category_ids
    
    if invalid_category_ids:
        invalid_list = sorted(invalid_category_ids)
        valid_list = sorted(valid_category_ids)
        raise ParticipantVisibleError(
            f'Submission contains invalid category_id values: {invalid_list}. '
            f'Valid category_ids are: {valid_list}'
        )
    
    # Get unique images and categories
    unique_images = sorted(gt_image_ids)
    unique_categories = sorted(valid_category_ids)
    
    # Build COCO ground truth format
    coco_gt = {
        'info': {'description': 'Kaggle Competition Ground Truth'},
        'licenses': [],
        'images': [{'id': int(img_id)} for img_id in unique_images],
        'categories': [{'id': int(cat_id), 'name': str(cat_id)} for cat_id in unique_categories],
        'annotations': []
    }
    
    # Add ground truth annotations
    # NOTE: COCO annotation IDs must start from 1, not 0, because pycocotools uses 0 to mean "no match"
    for ann_idx, (idx, row) in enumerate(gt_df.iterrows(), start=1):
        coco_gt['annotations'].append({
            'id': int(ann_idx),
            'image_id': int(row['image_id']),
            'category_id': int(row['category_id']),
            'bbox': [float(row['bbox_x']), float(row['bbox_y']), 
                    float(row['bbox_width']), float(row['bbox_height'])],
            'area': float(row['bbox_width'] * row['bbox_height']),
            'iscrowd': 0
        })
    
    # Build COCO detections format
    coco_dt = []
    for idx, row in pred_df.iterrows():
        coco_dt.append({
            'image_id': int(row['image_id']),
            'category_id': int(row['category_id']),
            'bbox': [float(row['bbox_x']), float(row['bbox_y']),
                    float(row['bbox_width']), float(row['bbox_height'])],
            'score': float(row['score'])
        })
    
    # Handle empty predictions
    if len(coco_dt) == 0:
        return 0.0
    
    # Suppress pycocotools verbose output
    original_stdout = sys.stdout
    sys.stdout = io.StringIO()
    
    try:
        # Create COCO ground truth object
        coco_gt_obj = COCO()
        coco_gt_obj.dataset = coco_gt
        coco_gt_obj.createIndex()
        
        # Load detections
        coco_dt_obj = coco_gt_obj.loadRes(coco_dt)
        
        # Run COCO evaluation
        coco_eval = COCOeval(coco_gt_obj, coco_dt_obj, 'bbox')
        coco_eval.evaluate()
        coco_eval.accumulate()
        coco_eval.summarize()
        
        # Extract mAP@[.50:.95] (first metric in stats)
        # stats[0] is AP @ IoU=0.50:0.95 | area=all | maxDets=100
        # Note: maxDets=100 means up to 100 detections per image are considered
        # Images with >100 annotations may not achieve perfect 1.0 score
        mAP = coco_eval.stats[0]
        
    finally:
        # Restore stdout
        sys.stdout = original_stdout
    
    # Return mAP, converting NaN to 0.0
    return float(mAP) if not np.isnan(mAP) else 0.0
